# Data Quality & Observability Framework

## Purpose
This notebook implements an enterprise-grade data quality framework with comprehensive audit logging, automated validation checks, and health scoring.

## Healthcare Analytics Parallel
In healthcare BI environments, data quality is mission-critical:
- **Clinical Data Validation**: Equivalent to validating patient identifiers, diagnosis codes, and medication records
- **Audit Trails**: Required for HIPAA compliance and Joint Commission standards
- **Data Trust**: Clinicians must trust data before making care decisions
- **Automated Monitoring**: Similar to clinical decision support alerts for data anomalies

## Implementation Components
1. **Audit Schema**: `mff_audit` database for all governance artifacts
2. **DQ Results Table**: Logs every validation check with pass/fail metrics
3. **Validation Checks**: 7 production-grade checks covering nulls, referential integrity, ranges, duplicates
4. **Health Scorecard**: Weighted composite score (0-100) for pipeline health
5. **Alerting**: Automatic warnings when CRITICAL checks fail

## Data Quality Checks Implemented
- **Null Rate Check**: Primary key columns must be 100% populated
- **Referential Integrity**: Foreign key relationships must be valid
- **Date Range Validation**: Data must fall within expected temporal boundaries
- **Revenue Sanity Check**: Business logic validation (price ranges)
- **Duplicate Detection**: Unique constraints on business keys
- **Cardinality Check**: Domain value validation (controlled vocabularies)

This framework can be scheduled to run after each ETL batch, providing continuous data observability.

In [0]:
# =============================================================================
# IMPORTS AND CONFIGURATION
# =============================================================================
# Healthcare Parallel: Similar to importing clinical validation libraries
# and configuring data quality rule engines in Epic Clarity environments

from pyspark.sql import functions as F
from pyspark.sql.types import *
from datetime import datetime
import warnings

# Configuration: Define validation thresholds
# In healthcare: These would be defined in data governance policies
CRITICAL_FAILURE_THRESHOLD = 0.01  # 1% failure rate triggers alert
WARNING_FAILURE_THRESHOLD = 0.05   # 5% failure rate triggers warning

# Define expected data boundaries for Maven Fuzzy Factory dataset
DATA_START_DATE = '2012-01-01'
DATA_END_DATE = '2016-01-01'

print("✓ Imports complete")
print(f"✓ Critical failure threshold: {CRITICAL_FAILURE_THRESHOLD * 100}%")
print(f"✓ Data boundary: {DATA_START_DATE} to {DATA_END_DATE}")

In [0]:
# =============================================================================
# CREATE AUDIT SCHEMA AND DQ RESULTS TABLE
# =============================================================================
# Healthcare Parallel: Similar to creating an audit database in Epic Reporting
# Database or SSRS admin database for tracking data validation history

# Create audit schema if it doesn't exist
spark.sql("CREATE DATABASE IF NOT EXISTS mff_audit")
print("✓ Created/verified mff_audit schema")

# Create DQ results table to log every validation check
# This is the audit trail for all data quality checks run against the platform
spark.sql("""
    CREATE TABLE IF NOT EXISTS mff_audit.dq_results (
        check_id STRING COMMENT 'Unique identifier for this check execution',
        check_name STRING COMMENT 'Name of the DQ check (e.g., null_rate_check)',
        table_name STRING COMMENT 'Fully qualified table name being validated',
        column_name STRING COMMENT 'Column being validated (NULL if table-level check)',
        records_checked BIGINT COMMENT 'Total number of records evaluated',
        records_failed BIGINT COMMENT 'Number of records that failed the check',
        failure_rate_pct DOUBLE COMMENT 'Percentage of records that failed (0-100)',
        severity STRING COMMENT 'CRITICAL, WARNING, or INFO',
        pipeline_stage STRING COMMENT 'bronze, silver, or gold',
        run_timestamp TIMESTAMP COMMENT 'When this check was executed',
        check_definition STRING COMMENT 'SQL or logic used for this check',
        passed BOOLEAN COMMENT 'True if failure rate below threshold',
        error_sample STRING COMMENT 'Sample of failed records for debugging'
    )
    USING DELTA
    COMMENT 'Data Quality check results audit log - equivalent to clinical data validation audit table in healthcare EDW'
""")

print("✓ Created/verified mff_audit.dq_results table")
print("\nTable ready to receive DQ check results")

# Show existing check history if any
history_count = spark.sql("SELECT COUNT(*) as cnt FROM mff_audit.dq_results").collect()[0]['cnt']
print(f"\nExisting DQ check history: {history_count:,} checks logged")

In [0]:
# =============================================================================
# DQ CHECK EXECUTION FUNCTION
# =============================================================================
# Healthcare Parallel: Reusable validation function similar to clinical
# data quality frameworks that standardize how checks are executed and logged

import uuid

def log_dq_check(check_name, table_name, column_name, records_checked, 
                 records_failed, severity, pipeline_stage, check_definition,
                 error_sample=None, failure_threshold=CRITICAL_FAILURE_THRESHOLD):
    """
    Log a data quality check result to the audit table.
    
    This function standardizes DQ check logging across all validation types.
    In healthcare BI, this is equivalent to logging clinical data validation
    results to an audit database for compliance and quality monitoring.
    
    Parameters:
    -----------
    check_name : str
        Name of the validation check (e.g., 'null_rate_check')
    table_name : str
        Fully qualified table name being validated
    column_name : str or None
        Column being validated (None for table-level checks)
    records_checked : int
        Total records evaluated
    records_failed : int
        Number of records that failed validation
    severity : str
        'CRITICAL', 'WARNING', or 'INFO'
    pipeline_stage : str
        'bronze', 'silver', or 'gold'
    check_definition : str
        SQL or logic used for this check
    error_sample : str or None
        Sample of failed records for debugging
    failure_threshold : float
        Threshold for determining pass/fail (default 1%)
    
    Returns:
    --------
    dict : Summary of check result including pass/fail status
    """
    
    check_id = str(uuid.uuid4())
    run_timestamp = datetime.now()
    
    # Calculate failure rate
    if records_checked > 0:
        failure_rate_pct = (records_failed / records_checked) * 100
    else:
        failure_rate_pct = 0.0
    
    # Determine pass/fail based on severity and threshold
    if severity == 'CRITICAL':
        passed = failure_rate_pct <= (failure_threshold * 100)
    elif severity == 'WARNING':
        passed = failure_rate_pct <= (WARNING_FAILURE_THRESHOLD * 100)
    else:  # INFO
        passed = True  # Informational checks always pass
    
    # Create result row
    result_data = [(
        check_id,
        check_name,
        table_name,
        column_name,
        records_checked,
        records_failed,
        round(failure_rate_pct, 4),
        severity,
        pipeline_stage,
        run_timestamp,
        check_definition,
        passed,
        error_sample
    )]
    
    # Define schema
    schema = StructType([
        StructField("check_id", StringType(), False),
        StructField("check_name", StringType(), False),
        StructField("table_name", StringType(), False),
        StructField("column_name", StringType(), True),
        StructField("records_checked", LongType(), False),
        StructField("records_failed", LongType(), False),
        StructField("failure_rate_pct", DoubleType(), False),
        StructField("severity", StringType(), False),
        StructField("pipeline_stage", StringType(), False),
        StructField("run_timestamp", TimestampType(), False),
        StructField("check_definition", StringType(), False),
        StructField("passed", BooleanType(), False),
        StructField("error_sample", StringType(), True)
    ])
    
    # Write to audit table
    result_df = spark.createDataFrame(result_data, schema)
    result_df.write.mode("append").saveAsTable("mff_audit.dq_results")
    
    # Return summary
    return {
        'check_name': check_name,
        'table_name': table_name,
        'column_name': column_name,
        'records_checked': records_checked,
        'records_failed': records_failed,
        'failure_rate_pct': round(failure_rate_pct, 4),
        'passed': passed,
        'severity': severity
    }

print("✓ DQ check logging function defined")
print("Ready to execute validation checks")

In [0]:
# =============================================================================
# CHECK 1: NULL RATE VALIDATION ON PRIMARY KEYS
# =============================================================================
# Healthcare Parallel: Equivalent to validating that Patient MRN, Encounter CSN,
# and Order IDs are never null in clinical data systems
# CRITICAL severity because missing primary keys break all downstream joins

print("\n" + "="*80)
print("EXECUTING: Null Rate Checks on Primary Key Columns")
print("="*80)

# Define primary key columns to check across Bronze tables
pk_checks = [
    ('mff_bronze.website_sessions', 'website_session_id', 'bronze'),
    ('mff_bronze.website_pageviews', 'website_pageview_id', 'bronze'),
    ('mff_bronze.orders', 'order_id', 'bronze'),
    ('mff_bronze.order_items', 'order_item_id', 'bronze'),
    ('mff_bronze.order_item_refunds', 'order_item_refund_id', 'bronze'),
    ('mff_bronze.products', 'product_id', 'bronze')
]

null_check_results = []

for table_name, pk_column, stage in pk_checks:
    # Count total records and null records
    check_query = f"""
        SELECT 
            COUNT(*) as total_records,
            SUM(CASE WHEN {pk_column} IS NULL THEN 1 ELSE 0 END) as null_records
        FROM {table_name}
    """
    
    result = spark.sql(check_query).collect()[0]
    total_records = result['total_records']
    null_records = result['null_records']
    
    # Log the check
    check_result = log_dq_check(
        check_name='null_rate_check',
        table_name=table_name,
        column_name=pk_column,
        records_checked=total_records,
        records_failed=null_records,
        severity='CRITICAL',
        pipeline_stage=stage,
        check_definition=f"Check that {pk_column} is never NULL",
        error_sample=None  # PKs should never be null, so no sample needed
    )
    
    null_check_results.append(check_result)
    
    # Print result
    status = "✓ PASS" if check_result['passed'] else "✗ FAIL"
    print(f"{status} | {table_name}.{pk_column}: {null_records:,} nulls / {total_records:,} records ({check_result['failure_rate_pct']}%)")

print(f"\n✓ Completed {len(null_check_results)} null rate checks on primary keys")

In [0]:
# =============================================================================
# CHECK 2: REFERENTIAL INTEGRITY VALIDATION
# =============================================================================
# Healthcare Parallel: Ensuring every Order has a valid Encounter, every
# Encounter has a valid Patient - fundamental to clinical data integrity
# CRITICAL severity because orphaned records indicate data pipeline failures

print("\n" + "="*80)
print("EXECUTING: Referential Integrity Checks")
print("="*80)

ref_integrity_results = []

# -------------------------------------------------------------------------
# CHECK 2a: Every order must have a valid website_session_id
# -------------------------------------------------------------------------
print("\nChecking: orders.website_session_id → website_sessions.website_session_id")

check_query = """
    SELECT COUNT(*) as orphaned_orders
    FROM mff_bronze.orders o
    LEFT JOIN mff_bronze.website_sessions ws 
        ON o.website_session_id = ws.website_session_id
    WHERE ws.website_session_id IS NULL
        AND o.website_session_id IS NOT NULL
"""

orphaned_orders = spark.sql(check_query).collect()[0]['orphaned_orders']
total_orders = spark.sql("SELECT COUNT(*) as cnt FROM mff_bronze.orders").collect()[0]['cnt']

# Get sample of orphaned records for debugging
if orphaned_orders > 0:
    sample_query = """
        SELECT o.order_id, o.website_session_id
        FROM mff_bronze.orders o
        LEFT JOIN mff_bronze.website_sessions ws 
            ON o.website_session_id = ws.website_session_id
        WHERE ws.website_session_id IS NULL
            AND o.website_session_id IS NOT NULL
        LIMIT 5
    """
    error_sample = str(spark.sql(sample_query).collect())
else:
    error_sample = None

check_result = log_dq_check(
    check_name='referential_integrity_check',
    table_name='mff_bronze.orders',
    column_name='website_session_id',
    records_checked=total_orders,
    records_failed=orphaned_orders,
    severity='CRITICAL',
    pipeline_stage='bronze',
    check_definition='Every order.website_session_id must exist in website_sessions',
    error_sample=error_sample
)

ref_integrity_results.append(check_result)
status = "✓ PASS" if check_result['passed'] else "✗ FAIL"
print(f"{status} | Orphaned orders: {orphaned_orders:,} / {total_orders:,} ({check_result['failure_rate_pct']}%)")

# -------------------------------------------------------------------------
# CHECK 2b: Every order_item must have a valid order_id
# -------------------------------------------------------------------------
print("\nChecking: order_items.order_id → orders.order_id")

check_query = """
    SELECT COUNT(*) as orphaned_items
    FROM mff_bronze.order_items oi
    LEFT JOIN mff_bronze.orders o
        ON oi.order_id = o.order_id
    WHERE o.order_id IS NULL
        AND oi.order_id IS NOT NULL
"""

orphaned_items = spark.sql(check_query).collect()[0]['orphaned_items']
total_items = spark.sql("SELECT COUNT(*) as cnt FROM mff_bronze.order_items").collect()[0]['cnt']

if orphaned_items > 0:
    sample_query = """
        SELECT oi.order_item_id, oi.order_id
        FROM mff_bronze.order_items oi
        LEFT JOIN mff_bronze.orders o
            ON oi.order_id = o.order_id
        WHERE o.order_id IS NULL
            AND oi.order_id IS NOT NULL
        LIMIT 5
    """
    error_sample = str(spark.sql(sample_query).collect())
else:
    error_sample = None

check_result = log_dq_check(
    check_name='referential_integrity_check',
    table_name='mff_bronze.order_items',
    column_name='order_id',
    records_checked=total_items,
    records_failed=orphaned_items,
    severity='CRITICAL',
    pipeline_stage='bronze',
    check_definition='Every order_item.order_id must exist in orders',
    error_sample=error_sample
)

ref_integrity_results.append(check_result)
status = "✓ PASS" if check_result['passed'] else "✗ FAIL"
print(f"{status} | Orphaned order items: {orphaned_items:,} / {total_items:,} ({check_result['failure_rate_pct']}%)")

print(f"\n✓ Completed {len(ref_integrity_results)} referential integrity checks")

In [0]:
# =============================================================================
# CHECK 3: DATE RANGE VALIDATION
# =============================================================================
# Healthcare Parallel: Ensuring service dates fall within valid ranges,
# no future-dated encounters, no procedures before patient birth date
# WARNING severity because date anomalies may be data entry errors vs pipeline failures

print("\n" + "="*80)
print("EXECUTING: Date Range Validation Checks")
print("="*80)

date_range_results = []

# Define tables and their date columns to validate
date_checks = [
    ('mff_bronze.website_sessions', 'created_at', 'bronze'),
    ('mff_bronze.website_pageviews', 'created_at', 'bronze'),
    ('mff_bronze.orders', 'created_at', 'bronze')
]

for table_name, date_column, stage in date_checks:
    print(f"\nValidating: {table_name}.{date_column}")
    
    # Check for records outside expected date range
    check_query = f"""
        SELECT 
            COUNT(*) as total_records,
            SUM(CASE 
                WHEN {date_column} < '{DATA_START_DATE}' 
                    OR {date_column} >= '{DATA_END_DATE}'
                THEN 1 
                ELSE 0 
            END) as invalid_dates,
            MIN({date_column}) as earliest_date,
            MAX({date_column}) as latest_date
        FROM {table_name}
    """
    
    result = spark.sql(check_query).collect()[0]
    total_records = result['total_records']
    invalid_dates = result['invalid_dates']
    earliest = result['earliest_date']
    latest = result['latest_date']
    
    # Get sample of invalid dates for debugging
    if invalid_dates > 0:
        sample_query = f"""
            SELECT *
            FROM {table_name}
            WHERE {date_column} < '{DATA_START_DATE}' 
                OR {date_column} >= '{DATA_END_DATE}'
            LIMIT 5
        """
        error_sample = f"Sample invalid dates: {spark.sql(sample_query).collect()}"
    else:
        error_sample = None
    
    check_result = log_dq_check(
        check_name='date_range_validation',
        table_name=table_name,
        column_name=date_column,
        records_checked=total_records,
        records_failed=invalid_dates,
        severity='WARNING',
        pipeline_stage=stage,
        check_definition=f"{date_column} must be between {DATA_START_DATE} and {DATA_END_DATE}",
        error_sample=error_sample
    )
    
    date_range_results.append(check_result)
    
    status = "✓ PASS" if check_result['passed'] else "✗ FAIL"
    print(f"{status} | Invalid dates: {invalid_dates:,} / {total_records:,} ({check_result['failure_rate_pct']}%)")
    print(f"     Actual date range: {earliest} to {latest}")

print(f"\n✓ Completed {len(date_range_results)} date range validation checks")

In [0]:
# =============================================================================
# CHECK 4: REVENUE SANITY VALIDATION
# =============================================================================
# Healthcare Parallel: Validating that charge amounts are within expected ranges,
# no negative balances (except refunds), procedure costs align with fee schedules
# WARNING severity because outliers may be legitimate (bulk orders, discounts)

print("\n" + "="*80)
print("EXECUTING: Revenue Sanity Checks")
print("="*80)

revenue_sanity_results = []

# -------------------------------------------------------------------------
# CHECK 4a: Order revenue must be between $1 and $500
# -------------------------------------------------------------------------
print("\nValidating: orders.price_usd should be between $1 and $500")

check_query = """
    SELECT 
        COUNT(*) as total_orders,
        SUM(CASE 
            WHEN price_usd < 1 OR price_usd > 500 
            THEN 1 
            ELSE 0 
        END) as invalid_prices,
        MIN(price_usd) as min_price,
        MAX(price_usd) as max_price,
        ROUND(AVG(price_usd), 2) as avg_price
    FROM mff_bronze.orders
"""

result = spark.sql(check_query).collect()[0]
total_orders = result['total_orders']
invalid_prices = result['invalid_prices']
min_price = result['min_price']
max_price = result['max_price']
avg_price = result['avg_price']

# Get sample of invalid prices
if invalid_prices > 0:
    sample_query = """
        SELECT order_id, price_usd, items_purchased
        FROM mff_bronze.orders
        WHERE price_usd < 1 OR price_usd > 500
        ORDER BY price_usd DESC
        LIMIT 5
    """
    error_sample = f"Sample invalid prices: {spark.sql(sample_query).collect()}"
else:
    error_sample = None

check_result = log_dq_check(
    check_name='revenue_sanity_check',
    table_name='mff_bronze.orders',
    column_name='price_usd',
    records_checked=total_orders,
    records_failed=invalid_prices,
    severity='WARNING',
    pipeline_stage='bronze',
    check_definition='price_usd must be between $1 and $500',
    error_sample=error_sample
)

revenue_sanity_results.append(check_result)

status = "✓ PASS" if check_result['passed'] else "✗ FAIL"
print(f"{status} | Invalid prices: {invalid_prices:,} / {total_orders:,} ({check_result['failure_rate_pct']}%)")
print(f"     Actual price range: ${min_price} to ${max_price} (avg: ${avg_price})")

# -------------------------------------------------------------------------
# CHECK 4b: Order items price must be positive
# -------------------------------------------------------------------------
print("\nValidating: order_items.price_usd must be positive")

check_query = """
    SELECT 
        COUNT(*) as total_items,
        SUM(CASE WHEN price_usd <= 0 THEN 1 ELSE 0 END) as negative_prices
    FROM mff_bronze.order_items
"""

result = spark.sql(check_query).collect()[0]
total_items = result['total_items']
negative_prices = result['negative_prices']

if negative_prices > 0:
    sample_query = """
        SELECT order_item_id, order_id, price_usd
        FROM mff_bronze.order_items
        WHERE price_usd <= 0
        LIMIT 5
    """
    error_sample = f"Sample negative prices: {spark.sql(sample_query).collect()}"
else:
    error_sample = None

check_result = log_dq_check(
    check_name='revenue_sanity_check',
    table_name='mff_bronze.order_items',
    column_name='price_usd',
    records_checked=total_items,
    records_failed=negative_prices,
    severity='CRITICAL',
    pipeline_stage='bronze',
    check_definition='price_usd must be greater than 0',
    error_sample=error_sample
)

revenue_sanity_results.append(check_result)

status = "✓ PASS" if check_result['passed'] else "✗ FAIL"
print(f"{status} | Negative/zero prices: {negative_prices:,} / {total_items:,} ({check_result['failure_rate_pct']}%)")

print(f"\n✓ Completed {len(revenue_sanity_results)} revenue sanity checks")

In [0]:
# =============================================================================
# CHECK 5: DUPLICATE DETECTION
# =============================================================================
# Healthcare Parallel: Detecting duplicate patient registrations, duplicate orders,
# duplicate lab results - critical for accurate patient counts and metrics
# CRITICAL severity because duplicates cause overcounting in all downstream analytics

print("\n" + "="*80)
print("EXECUTING: Duplicate Detection Checks")
print("="*80)

duplicate_detection_results = []

# Define uniqueness checks
uniqueness_checks = [
    ('mff_bronze.website_sessions', 'website_session_id', 'bronze'),
    ('mff_bronze.orders', 'order_id', 'bronze'),
    ('mff_bronze.order_items', 'order_item_id', 'bronze')
]

for table_name, unique_column, stage in uniqueness_checks:
    print(f"\nChecking uniqueness: {table_name}.{unique_column}")
    
    # Count duplicates
    check_query = f"""
        SELECT 
            COUNT(*) as total_records,
            COUNT(DISTINCT {unique_column}) as unique_values,
            COUNT(*) - COUNT(DISTINCT {unique_column}) as duplicate_records
        FROM {table_name}
    """
    
    result = spark.sql(check_query).collect()[0]
    total_records = result['total_records']
    unique_values = result['unique_values']
    duplicate_records = result['duplicate_records']
    
    # Get sample of duplicates if any exist
    if duplicate_records > 0:
        sample_query = f"""
            SELECT {unique_column}, COUNT(*) as occurrence_count
            FROM {table_name}
            GROUP BY {unique_column}
            HAVING COUNT(*) > 1
            ORDER BY occurrence_count DESC
            LIMIT 5
        """
        error_sample = f"Sample duplicates: {spark.sql(sample_query).collect()}"
    else:
        error_sample = None
    
    check_result = log_dq_check(
        check_name='duplicate_detection',
        table_name=table_name,
        column_name=unique_column,
        records_checked=total_records,
        records_failed=duplicate_records,
        severity='CRITICAL',
        pipeline_stage=stage,
        check_definition=f'{unique_column} must be unique (no duplicates)',
        error_sample=error_sample
    )
    
    duplicate_detection_results.append(check_result)
    
    status = "✓ PASS" if check_result['passed'] else "✗ FAIL"
    print(f"{status} | Duplicate records: {duplicate_records:,} / {total_records:,} ({check_result['failure_rate_pct']}%)")
    print(f"     Unique values: {unique_values:,}")

print(f"\n✓ Completed {len(duplicate_detection_results)} duplicate detection checks")

In [0]:
# =============================================================================
# CHECK 6: CARDINALITY VALIDATION
# =============================================================================
# Healthcare Parallel: Validating that diagnosis codes come from ICD-10 vocabulary,
# admission types from controlled list, physician specialties from standard taxonomy
# WARNING severity because new values may be legitimate vs data quality issues

print("\n" + "="*80)
print("EXECUTING: Cardinality Validation Checks")
print("="*80)

cardinality_results = []

# -------------------------------------------------------------------------
# CHECK 6a: utm_source should have fewer than 20 distinct values
# -------------------------------------------------------------------------
print("\nValidating: website_sessions.utm_source cardinality")

check_query = """
    SELECT 
        COUNT(*) as total_sessions,
        COUNT(DISTINCT utm_source) as distinct_sources
    FROM mff_bronze.website_sessions
    WHERE utm_source IS NOT NULL
"""

result = spark.sql(check_query).collect()[0]
total_sessions = result['total_sessions']
distinct_sources = result['distinct_sources']

# Check if cardinality exceeds expected threshold
expected_max_cardinality = 20
cardinality_exceeded = 1 if distinct_sources > expected_max_cardinality else 0

# Get distribution of sources
if distinct_sources > 0:
    distribution_query = """
        SELECT 
            utm_source,
            COUNT(*) as session_count,
            ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) as pct_of_total
        FROM mff_bronze.website_sessions
        WHERE utm_source IS NOT NULL
        GROUP BY utm_source
        ORDER BY session_count DESC
    """
    distribution = spark.sql(distribution_query).collect()
    error_sample = f"Source distribution: {distribution[:10]}"
else:
    error_sample = None

check_result = log_dq_check(
    check_name='cardinality_validation',
    table_name='mff_bronze.website_sessions',
    column_name='utm_source',
    records_checked=total_sessions,
    records_failed=cardinality_exceeded * total_sessions if cardinality_exceeded else 0,
    severity='WARNING',
    pipeline_stage='bronze',
    check_definition=f'utm_source should have fewer than {expected_max_cardinality} distinct values',
    error_sample=error_sample
)

cardinality_results.append(check_result)

status = "✓ PASS" if distinct_sources <= expected_max_cardinality else "✗ WARNING"
print(f"{status} | Distinct utm_source values: {distinct_sources} (threshold: {expected_max_cardinality})")
print(f"     Total sessions: {total_sessions:,}")

# -------------------------------------------------------------------------
# CHECK 6b: device_type should have expected values (mobile, desktop, etc.)
# -------------------------------------------------------------------------
print("\nValidating: website_sessions.device_type vocabulary")

expected_device_types = ['desktop', 'mobile', 'tablet']  # Expected controlled vocabulary

check_query = """
    SELECT 
        COUNT(*) as total_sessions,
        SUM(CASE 
            WHEN LOWER(device_type) NOT IN ('desktop', 'mobile', 'tablet')
            THEN 1 
            ELSE 0 
        END) as unexpected_devices
    FROM mff_bronze.website_sessions
    WHERE device_type IS NOT NULL
"""

result = spark.sql(check_query).collect()[0]
total_sessions_with_device = result['total_sessions']
unexpected_devices = result['unexpected_devices']

if unexpected_devices > 0:
    sample_query = """
        SELECT DISTINCT device_type, COUNT(*) as count
        FROM mff_bronze.website_sessions
        WHERE LOWER(device_type) NOT IN ('desktop', 'mobile', 'tablet')
            AND device_type IS NOT NULL
        GROUP BY device_type
    """
    error_sample = f"Unexpected device types: {spark.sql(sample_query).collect()}"
else:
    error_sample = None

check_result = log_dq_check(
    check_name='cardinality_validation',
    table_name='mff_bronze.website_sessions',
    column_name='device_type',
    records_checked=total_sessions_with_device,
    records_failed=unexpected_devices,
    severity='WARNING',
    pipeline_stage='bronze',
    check_definition=f'device_type should be one of: {", ".join(expected_device_types)}',
    error_sample=error_sample
)

cardinality_results.append(check_result)

status = "✓ PASS" if check_result['passed'] else "✗ WARNING"
print(f"{status} | Unexpected device types: {unexpected_devices:,} / {total_sessions_with_device:,} ({check_result['failure_rate_pct']}%)")

print(f"\n✓ Completed {len(cardinality_results)} cardinality validation checks")

In [0]:
# =============================================================================
# GENERATE DATA QUALITY SCORECARD
# =============================================================================
# Healthcare Parallel: Executive dashboard showing overall EDW data quality score,
# similar to how clinical quality scorecards aggregate HEDIS measures
# Provides single composite metric for data trust and governance reporting

print("\n" + "="*80)
print("DATA QUALITY SCORECARD - OVERALL PIPELINE HEALTH")
print("="*80)

# Query all checks from current run (last 5 minutes)
scorecard_query = """
    SELECT 
        severity,
        COUNT(*) as total_checks,
        SUM(CASE WHEN passed = TRUE THEN 1 ELSE 0 END) as passed_checks,
        SUM(CASE WHEN passed = FALSE THEN 1 ELSE 0 END) as failed_checks,
        ROUND(AVG(CASE WHEN passed = TRUE THEN 100 ELSE 0 END), 2) as pass_rate_pct
    FROM mff_audit.dq_results
    WHERE run_timestamp >= CURRENT_TIMESTAMP() - INTERVAL 5 MINUTES
    GROUP BY severity
    ORDER BY 
        CASE severity
            WHEN 'CRITICAL' THEN 1
            WHEN 'WARNING' THEN 2
            WHEN 'INFO' THEN 3
        END
"""

scorecard_df = spark.sql(scorecard_query)
scorecard_results = scorecard_df.collect()

print("\n" + "-"*80)
print(f"{'SEVERITY':<12} {'TOTAL':<10} {'PASSED':<10} {'FAILED':<10} {'PASS RATE':<12}")
print("-"*80)

total_weighted_score = 0
total_weight = 0

# Weighted scoring: CRITICAL = 50%, WARNING = 30%, INFO = 20%
severity_weights = {
    'CRITICAL': 0.50,
    'WARNING': 0.30,
    'INFO': 0.20
}

for row in scorecard_results:
    severity = row['severity']
    total = row['total_checks']
    passed = row['passed_checks']
    failed = row['failed_checks']
    pass_rate = row['pass_rate_pct']
    
    # Apply weighting
    weight = severity_weights.get(severity, 0)
    weighted_contribution = pass_rate * weight
    total_weighted_score += weighted_contribution
    total_weight += weight
    
    # Format output with status indicator
    status_icon = "✓" if pass_rate == 100 else "⚠" if pass_rate >= 80 else "✗"
    print(f"{status_icon} {severity:<10} {total:<10} {passed:<10} {failed:<10} {pass_rate}%")

print("-"*80)

# Calculate overall health score
if total_weight > 0:
    overall_health_score = round(total_weighted_score / total_weight, 2)
else:
    overall_health_score = 0.0

print(f"\n{'='*80}")
print(f"OVERALL PIPELINE HEALTH SCORE: {overall_health_score}/100")
print(f"{'='*80}")

# Interpret score
if overall_health_score >= 95:
    health_status = "EXCELLENT"
    health_icon = "✓"
    health_color = "GREEN"
elif overall_health_score >= 85:
    health_status = "GOOD"
    health_icon = "✓"
    health_color = "GREEN"
elif overall_health_score >= 70:
    health_status = "FAIR"
    health_icon = "⚠"
    health_color = "AMBER"
else:
    health_status = "POOR"
    health_icon = "✗"
    health_color = "RED"

print(f"\nHealth Status: {health_icon} {health_status} ({health_color})")
print(f"\nScoring Methodology:")
print(f"  • CRITICAL checks weighted at 50% (data integrity, referential integrity)")
print(f"  • WARNING checks weighted at 30% (business logic, ranges)")
print(f"  • INFO checks weighted at 20% (informational metrics)")
print(f"\nThis score represents data trustworthiness for downstream analytics.")
print(f"In healthcare BI: Similar to composite quality score for clinical data.")

# Query for detailed failed checks
failed_checks_query = """
    SELECT 
        check_name,
        table_name,
        column_name,
        severity,
        records_checked,
        records_failed,
        failure_rate_pct
    FROM mff_audit.dq_results
    WHERE passed = FALSE
        AND run_timestamp >= CURRENT_TIMESTAMP() - INTERVAL 5 MINUTES
    ORDER BY 
        CASE severity WHEN 'CRITICAL' THEN 1 WHEN 'WARNING' THEN 2 ELSE 3 END,
        failure_rate_pct DESC
"""

failed_checks_df = spark.sql(failed_checks_query)
failed_count = failed_checks_df.count()

if failed_count > 0:
    print(f"\n\n{'⚠'*3} FAILED CHECKS REQUIRING ATTENTION {'⚠'*3}")
    print("-"*80)
    failed_checks_df.show(truncate=False)
else:
    print(f"\n✓ All data quality checks passed successfully!")

print(f"\n✓ DQ Scorecard generation complete")

In [0]:
# =============================================================================
# ALERTING: RAISE WARNINGS FOR CRITICAL FAILURES
# =============================================================================
# Healthcare Parallel: Automated alerts for data quality failures - equivalent to
# clinical decision support alerts, Epic Sentry notifications, or HL7 interface
# error notifications that require immediate attention

print("\n" + "="*80)
print("CRITICAL FAILURE ALERTING")
print("="*80)

# Query for CRITICAL checks that failed above threshold
critical_failures_query = f"""
    SELECT 
        check_name,
        table_name,
        column_name,
        records_checked,
        records_failed,
        failure_rate_pct,
        check_definition,
        error_sample
    FROM mff_audit.dq_results
    WHERE severity = 'CRITICAL'
        AND passed = FALSE
        AND failure_rate_pct > {CRITICAL_FAILURE_THRESHOLD * 100}
        AND run_timestamp >= CURRENT_TIMESTAMP() - INTERVAL 5 MINUTES
    ORDER BY failure_rate_pct DESC
"""

critical_failures_df = spark.sql(critical_failures_query)
critical_failures = critical_failures_df.collect()

if len(critical_failures) > 0:
    print(f"\n❌ ALERT: {len(critical_failures)} CRITICAL check(s) failed above {CRITICAL_FAILURE_THRESHOLD * 100}% threshold!\n")
    print("="*80)
    
    for idx, failure in enumerate(critical_failures, 1):
        print(f"\n[ALERT #{idx}]")
        print(f"  Check: {failure['check_name']}")
        print(f"  Table: {failure['table_name']}")
        print(f"  Column: {failure['column_name']}")
        print(f"  Definition: {failure['check_definition']}")
        print(f"  Records Checked: {failure['records_checked']:,}")
        print(f"  Records Failed: {failure['records_failed']:,}")
        print(f"  Failure Rate: {failure['failure_rate_pct']}%")
        
        if failure['error_sample']:
            print(f"  Error Sample: {failure['error_sample'][:200]}...")  # Truncate long samples
        
        print("-"*80)
    
    # In production, this would:
    # 1. Send email/Slack notification to data engineering team
    # 2. Create incident ticket in ServiceNow/Jira
    # 3. Pause downstream Gold layer jobs until resolved
    # 4. Log to central monitoring system (Datadog, Splunk, etc.)
    
    print("\n⚠ RECOMMENDED ACTIONS:")
    print("  1. Investigate root cause of data quality failures")
    print("  2. Check upstream data sources for issues")
    print("  3. Review recent pipeline changes or deployments")
    print("  4. Do NOT promote data to Gold layer until resolved")
    print("  5. Notify downstream consumers of potential data issues")
    
    # Raise Python warning to make it visible in notebook output
    warnings.warn(
        f"CRITICAL DATA QUALITY FAILURES DETECTED: {len(critical_failures)} check(s) failed. "
        f"Review mff_audit.dq_results for details.",
        UserWarning
    )
    
else:
    print("\n✓ No critical failures detected above threshold.")
    print(f"\nAll CRITICAL checks passed or failed below {CRITICAL_FAILURE_THRESHOLD * 100}% threshold.")
    print("Data pipeline is healthy and ready for downstream processing.")

print("\n" + "="*80)
print("DATA QUALITY FRAMEWORK EXECUTION COMPLETE")
print("="*80)
print(f"\nAudit Trail: All results logged to mff_audit.dq_results")
print(f"Query results: SELECT * FROM mff_audit.dq_results ORDER BY run_timestamp DESC")
print(f"\nIn healthcare BI context:")
print(f"  • This framework ensures clinical data meets quality standards")
print(f"  • Audit trail supports HIPAA compliance and Joint Commission requirements")
print(f"  • Automated monitoring prevents bad data from reaching dashboards")
print(f"  • Similar to Epic Clarity validation rules and SSRS data quality checks")